# Estuarine and Coastal Environmental Measurements (Basque Country 1995–2014) Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR² Estuarine and Coastal Environmental Measurements dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described using a [Croissant schema](https://mlcommons.org/croissant/), accessible at:

`https://sen.science/doi/10.71728/senscience.ah5f-b5yk/fair2.json`

In [ ]:
# Install mlcroissant if needed
!pip install mlcroissant

## 1. Data Loading

Load Croissant metadata and inspect high-level dataset details.

In [ ]:
import mlcroissant as mlc
import pandas as pd
from pprint import pprint

# Define Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.ah5f-b5yk/fair2.json'

# Load dataset via Croissant schema (no need to subscript .metadata; use its attributes)
dataset = mlc.Dataset(croissant_url)
print(f"Loaded dataset: {dataset.metadata.name}\n")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {dataset.metadata.datePublished}")

## 2. Data Overview

Review available record sets, fields, and their unique `@id`s (identifiers) as defined in the Croissant metadata.

In [ ]:
# List all record sets with their @id and human-readable name
record_sets = dataset.metadata.record_sets
print(f"Number of record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name if hasattr(rs, 'name') else '[No name]'}")
    print(f"  Description: {rs.description if hasattr(rs, 'description') else '[No description]'}\n")

# For each record set, list sample field @id and name
for rs in record_sets:
    print(f"Fields for RecordSet {rs.id}:")
    fields = rs.fields
    if fields:
        for f in fields[:5]:  # Show up to 5 fields per record set
            print(f"  Field @id: {f.id}")
            print(f"    Name: {f.name if hasattr(f, 'name') else '[No name]'}")
            print(f"    Data type: {f.data_type if hasattr(f, 'data_type') else '[Not specified]'}")
    else:
        print(f"  No fields defined in this record set.")
    print()

### Example: Show a small sample of records for one record set
Retrieve the first 2 records from the first record set. All record set references use their `@id`.

In [ ]:
# Pick one record set for demonstration. You may adjust the index for another set.
if len(record_sets) == 0:
    raise ValueError("No record sets are defined in this dataset metadata.")
first_rs = record_sets[0]
first_rs_id = first_rs.id

print(f"First record set: {first_rs_id}\n")
for i, rec in enumerate(dataset.records(record_set=first_rs_id)):
    pprint(rec)
    if i >= 1:  # Show two records only
        break

## 3. Data Extraction

Extract all records from each record set using their `@id` into pandas DataFrames for flexible analysis.

In [ ]:
# Gather all record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]

dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded RecordSet {rs_id}: {df.shape[0]} records, {df.shape[1]} columns.")
    else:
        print(f"RecordSet {rs_id} returned no records.")

# Show DataFrame columns for the first (main) record set
main_rs_id = record_set_ids[0]
print(f"\nColumns in '{main_rs_id}':")
print(list(dataframes[main_rs_id].columns))
dataframes[main_rs_id].head()

## 4. Exploratory Data Analysis (EDA)

Let's select one numeric field (`@id`) from the first record set for exploration, filter for larger values, normalize, and group by a categorical field if available.

You can adapt the field `@id`s below based on the output of step 3. All access is by `@id`, not by label.

In [ ]:
# Example: select a numeric field from the main record set
main_df = dataframes[main_rs_id]

# Find a likely numeric field (use the fields list from metadata)
candidate_numeric_fields = []
for f in dataset.metadata.record_sets[0].fields:
    if hasattr(f, 'data_type') and f.data_type in ['schema:Integer', 'schema:Float', 'schema:Number']:
        candidate_numeric_fields.append(f.id)

if not candidate_numeric_fields:
    raise ValueError("No numeric fields found in the first record set.")
numeric_field_id = candidate_numeric_fields[0]
print(f"Using numeric field: {numeric_field_id}")

# Filter for values greater than a threshold
threshold = main_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(main_df[numeric_field_id]) else 10
filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold]
print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} selected.")
print(filtered_df[[numeric_field_id]].head())

# Normalize
filtered_df.loc[:, f"{numeric_field_id}_normalized"] = (
    pd.to_numeric(filtered_df[numeric_field_id], errors='coerce') - pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').mean()
) / pd.to_numeric(filtered_df[numeric_field_id], errors='coerce').std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try grouping by a non-numeric field
candidate_group_fields = []
for f in dataset.metadata.record_sets[0].fields:
    if hasattr(f, 'data_type') and f.data_type.startswith('schema:') and ('Text' in f.data_type or 'Category' in f.data_type or 'String' in f.data_type):
        candidate_group_fields.append(f.id)
if candidate_group_fields:
    group_field = candidate_group_fields[0]
    print(f"\nGrouping by: {group_field}")
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(grouped_df.head())
else:
    print("No categorical field available for grouping.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field and its normalized form.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
filtered_df[numeric_field_id].hist(ax=axs[0], bins=30)
axs[0].set_title(f"Distribution of {numeric_field_id}")
axs[0].set_xlabel(numeric_field_id)

filtered_df[f"{numeric_field_id}_normalized"].hist(ax=axs[1], bins=30)
axs[1].set_title(f"Distribution of {numeric_field_id} (Normalized)")
axs[1].set_xlabel(f"{numeric_field_id} normalized")

plt.tight_layout()
plt.show()


## 6. Conclusion

In this notebook, we've demonstrated how to load, explore, and process the FAIR² environmental measurements dataset using `mlcroissant`. We extracted record sets by their `@id`, examined fields and columns, filtered and normalized data, grouped by categorical fields, and visualized numeric distributions.

**Key takeaways:**
- The Croissant format enables transparent, reproducible access to rich, multi-table environmental monitoring datasets.
- All references to subsets and fields are performed via unique `@id` values, ensuring consistent linkage to the metadata schema.
- With pandas and standard Python, you can readily filter, transform, and visualize Croissant-style datasets for scientific discovery.